In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import lightgbm as lgb
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


In [ ]:

# Exploratory Data Analysis
print("=== Missing Values (Train) ===")
print(train.isnull().sum())
print("\n=== Target (demand) Distribution ===")
print(train['demand'].describe())
print("\n=== Unique Days ===")
print("Train days:", sorted(train.day.unique()))
print("Test days:", sorted(test.day.unique()))
print("\n=== Weather Distribution ===")
print(train.Weather.value_counts())
print("\n=== RoadType Distribution ===")
print(train.RoadType.value_counts())


In [ ]:

def feature_engineering(df):
    df = df.copy()
    
    # Parse timestamp into hour and minute
    df['hour'] = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_of_day'] = df['hour'] * 60 + df['minute']
    
    # Cyclical encoding for time (captures periodicity)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['time_sin'] = np.sin(2 * np.pi * df['time_of_day'] / (24*60))
    df['time_cos'] = np.cos(2 * np.pi * df['time_of_day'] / (24*60))
    
    # Rush hour flags
    df['is_morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_evening_rush'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)
    df['is_midday'] = ((df['hour'] >= 11) & (df['hour'] <= 14)).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    
    # Road features encoding
    roadtype_map = {'Residential': 1, 'Street': 2, 'Highway': 3}
    df['RoadType_enc'] = df['RoadType'].map(roadtype_map).fillna(0)
    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)
    
    # Weather encoding (ordinal: higher = better visibility/driving)
    weather_map = {'Sunny': 3, 'Rainy': 2, 'Foggy': 1, 'Snowy': 0}
    df['Weather_enc'] = df['Weather'].map(weather_map).fillna(1.5)
    
    # Temperature - fill missing with mean, add polynomial
    temp_mean = df['Temperature'].mean()
    df['Temperature_filled'] = df['Temperature'].fillna(temp_mean)
    df['temp_squared'] = df['Temperature_filled'] ** 2
    
    return df

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

print("Feature engineering complete!")
print("New features added:", [c for c in train_fe.columns if c not in train.columns])


In [ ]:

# Target Encoding: Geohash and Timestamp Statistics
# These are the most powerful features since geohash encodes location demand patterns

# Geohash-level statistics
geo_stats = train.groupby('geohash')['demand'].agg(['mean', 'std', 'median', 'min', 'max']).reset_index()
geo_stats.columns = ['geohash', 'geo_mean_demand', 'geo_std_demand', 'geo_median_demand', 'geo_min_demand', 'geo_max_demand']

# Geohash + Timestamp combined (captures location-time interactions)
geo_ts_stats = train.groupby(['geohash', 'timestamp'])['demand'].agg(['mean', 'std']).reset_index()
geo_ts_stats.columns = ['geohash', 'timestamp', 'geo_ts_mean', 'geo_ts_std']

# Timestamp-level stats
ts_stats = train.groupby('timestamp')['demand'].agg(['mean', 'std']).reset_index()
ts_stats.columns = ['timestamp', 'ts_mean', 'ts_std']

# Hour-level stats
train_fe['hour_str'] = train_fe['hour'].astype(str)
hour_stats = train_fe.groupby('hour')['demand'].agg(['mean', 'std']).reset_index()
hour_stats.columns = ['hour', 'hour_mean', 'hour_std']

def merge_stats(df_fe):
    df_fe = df_fe.merge(geo_stats, on='geohash', how='left')
    df_fe = df_fe.merge(geo_ts_stats, on=['geohash', 'timestamp'], how='left')
    df_fe = df_fe.merge(ts_stats, on='timestamp', how='left')
    df_fe = df_fe.merge(hour_stats, on='hour', how='left')
    
    global_mean = train['demand'].mean()
    global_std = train['demand'].std()
    for col in ['geo_mean_demand', 'geo_median_demand', 'geo_ts_mean']:
        df_fe[col] = df_fe[col].fillna(global_mean)
    for col in ['geo_std_demand', 'geo_ts_std']:
        df_fe[col] = df_fe[col].fillna(global_std)
    return df_fe

train_fe = merge_stats(train_fe)
test_fe = merge_stats(test_fe)

print("Target encoding complete!")
print("Geo-ts mean coverage:", geo_ts_stats.shape[0], "combinations")


In [ ]:

features = [
    'hour', 'minute', 'time_of_day', 'hour_sin', 'hour_cos', 'time_sin', 'time_cos',
    'is_morning_rush', 'is_evening_rush', 'is_midday', 'is_night',
    'RoadType_enc', 'LargeVehicles_enc', 'Landmarks_enc',
    'Weather_enc', 'Temperature_filled', 'temp_squared',
    'NumberofLanes', 'day',
    'geo_mean_demand', 'geo_std_demand', 'geo_median_demand', 'geo_min_demand', 'geo_max_demand',
    'geo_ts_mean', 'geo_ts_std',
    'ts_mean', 'ts_std',
    'hour_mean', 'hour_std'
]

X_train = train_fe[features]
y_train = train_fe['demand']
X_test = test_fe[features]

print("Training features:", len(features))
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:

# 5-Fold Cross-Validation with LightGBM
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_tr, y_tr, 
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / 5
    
    fold_r2 = r2_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold+1} R2: {fold_r2:.4f}  |  Score: {max(0, 100*fold_r2):.2f}')

oof_r2 = r2_score(y_train, oof_preds)
print(f'\n=== Overall OOF R2: {oof_r2:.4f} ===')
print(f'=== Competition Score: {max(0, 100*oof_r2):.2f} / 100 ===')


In [ ]:

# Clip predictions to valid range [0, 1]
test_preds_clipped = np.clip(test_preds, 0, 1)

# Create submission file
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_preds_clipped
})

submission.to_csv('submission.csv', index=False)
print("Submission file saved!")
print("Shape:", submission.shape)
print("\nFirst 5 rows:")
print(submission.head())
print("\nDemand statistics:")
print(submission['demand'].describe())


In [ ]:

# Feature Importance Analysis
import matplotlib.pyplot as plt

feat_imp = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feat_imp.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
plt.barh(feat_imp.head(15)['feature'][::-1], feat_imp.head(15)['importance'][::-1])
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importances (LightGBM)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("Feature importance plot saved!")
